# Pipeline benchmark (fixed CV)

Compare **6** tuned pipelines on **train** data with frozen hyperparameters from `data/processed/tuned/<model_id>.json`.

| Pipeline | Description |
|----------|-------------|
| `mspc_lr` | MSPC + elastic-net logistic |
| `mspc_rf` | MSPC + Random Forest |
| `xgb_mspc` | MSPC + XGBoost |
| `rf_k_lr` | RF top-K + elastic-net logistic |
| `rf_k_rf` | RF top-K + Random Forest |
| `rf_k_knn` | RF top-K + KNN |

Run GridSearchCV first: each notebook in `tuning/` (e.g. `tuning/mspc_lr.ipynb`).

**Primary metric: ROC AUC (CV).** Leaderboards sorted by `mean_roc_auc` descending; holdout by `roc_auc` descending.

**CV:** repeated stratified 5×5 on the 80% train split. **Holdout:** 20% test (reporting only).

In [1]:
import importlib
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reload after editing scripts/ (avoids stale pipeline modules in kernel).
import scripts.benchmark_models as _bm
import scripts.mspc_features as _mf
import scripts.secom_pipelines as _sp

importlib.reload(_mf)
importlib.reload(_sp)
importlib.reload(_bm)

from scripts.benchmark_models import (
    BENCHMARK_RESULTS_PATH,
    build_benchmark_pipelines,
    run_holdout_benchmark,
    run_pipeline_benchmark,
    save_benchmark_results,
)
from scripts.secom_utils import load_all_tuned_params
from scripts.secom_pipelines import (
    TARGET_COL,
    feature_columns,
    load_mart,
    split_train_test,
)



In [2]:
tuned = load_all_tuned_params()
for model_id, payload in tuned.items():
    summary = payload.get("cv_summary", {})
    roc = summary.get("mean_roc_auc", "n/a")
    print(model_id, "mean_roc_auc=", roc, payload.get("grid_search_best_params", {}))

1.0 0.95 n_components=10


In [3]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL].astype(int)
print(len(X_train), len(X_test), y_train.mean())

1253 314 0.06624102154828412


In [4]:
pipelines = build_benchmark_pipelines(tuned)
list(pipelines.keys())

['mspc_lr',
 'mspc_knn',
 'mspc_rf',
 'xgb_mspc',
 'rf_k_lr_k10',
 'rf_k_knn_k10',
 'rf_k_rf_k10',
 'xgb_rf_k_k10',
 'rf_k_lr_k15',
 'rf_k_knn_k15',
 'rf_k_rf_k15',
 'xgb_rf_k_k15',
 'rf_k_lr_k20',
 'rf_k_knn_k20',
 'rf_k_rf_k20',
 'xgb_rf_k_k20',
 'rf_k_lr_k30',
 'rf_k_knn_k30',
 'rf_k_rf_k30',
 'xgb_rf_k_k30']

In [5]:
leaderboard = run_pipeline_benchmark(pipelines, X_train, y_train, show_progress=True)
display(leaderboard)

Benchmark: 20 pipelines x 25 folds


Benchmark pipeline:   0%|          | 0/20 [00:00<?, ?it/s]

CV mspc_lr:   0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  mspc_lr: mean ROC AUC 0.689 (±0.060)


...

  rf_k_rf_k30: mean ROC AUC 0.721 (±0.044)


CV xgb_rf_k_k30:   0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  xgb_rf_k_k30: mean ROC AUC 0.686 (±0.059)


,pipeline,mean_ber_percent,std_ber_percent,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
0,rf_k_rf_k30,41.536639,4.361043,23.661765,9.160171,93.264957,1.789900,0.720646,0.044171,0.191169,0.043365
1,rf_k_rf_k20,40.202174,4.478028,28.279412,8.989806,91.316239,2.178129,0.714456,0.043103,0.195705,0.043934
2,rf_k_rf_k15,39.207328,4.183774,32.132353,8.960940,89.452991,3.050025,0.710502,0.047384,0.209886,0.058793
3,rf_k_rf_k10,39.423831,4.495702,34.588235,9.287973,86.564103,3.429641,0.708852,0.047719,0.204165,0.057485
4,mspc_rf,40.166730,4.567118,27.632353,9.217128,92.034188,1.630128,0.701939,0.066803,0.201604,0.062382
5,xgb_mspc,43.226496,3.953677,19.000000,8.132047,94.547009,1.275081,0.693545,0.057053,0.184628,0.054310
6,mspc_lr,36.824786,6.351388,43.000000,13.216686,83.350427,2.522396,0.689412,0.060378,0.191843,0.060734
7,xgb_rf_k_k30,47.285508,2.566272,6.779412,5.229466,98.649573,0.647320,0.685676,0.058679,0.178529,0.048604
8,xgb_rf_k_k20,46.634490,3.551917,8.235294,7.074126,98.495726,0.746286,0.682412,0.052891,0.186081,0.051809
9,rf_k_lr_k30,37.381222,4.726178,51.852941,11.917882,73.384615,5.223502,0.679981,0.061516,0.176315,0.050105


In [6]:
holdout = run_holdout_benchmark(
    pipelines, X_train, y_train, X_test, y_test, show_progress=True
)
display(holdout)

Holdout: 20 pipelines (reporting only)
  mspc_lr: holdout ROC AUC 0.653
  mspc_knn: holdout ROC AUC 0.653
  mspc_rf: holdout ROC AUC 0.730
  xgb_mspc: holdout ROC AUC 0.701
  rf_k_lr_k10: holdout ROC AUC 0.699
  rf_k_knn_k10: holdout ROC AUC 0.710
  rf_k_rf_k10: holdout ROC AUC 0.748
  xgb_rf_k_k10: holdout ROC AUC 0.729
  rf_k_lr_k15: holdout ROC AUC 0.727
  rf_k_knn_k15: holdout ROC AUC 0.667
  rf_k_rf_k15: holdout ROC AUC 0.758
  xgb_rf_k_k15: holdout ROC AUC 0.723
  rf_k_lr_k20: holdout ROC AUC 0.692
  rf_k_knn_k20: holdout ROC AUC 0.648
  rf_k_rf_k20: holdout ROC AUC 0.756
  xgb_rf_k_k20: holdout ROC AUC 0.755
  rf_k_lr_k30: holdout ROC AUC 0.692
  rf_k_knn_k30: holdout ROC AUC 0.676
  rf_k_rf_k30: holdout ROC AUC 0.744
  xgb_rf_k_k30: holdout ROC AUC 0.719


,pipeline,true_positive_rate,true_negative_rate,balanced_accuracy,ber_percent,true_positive_percent,true_negative_percent,confusion_matrix,roc_auc,pr_auc
0,rf_k_rf_k15,0.285714,0.897611,0.591663,40.833740,28.571429,89.761092,"[[263, 30], [15, 6]]",0.758004,0.201393
1,rf_k_rf_k20,0.285714,0.914676,0.600195,39.980497,28.571429,91.467577,"[[268, 25], [15, 6]]",0.755891,0.171322
2,xgb_rf_k_k20,0.047619,0.989761,0.518690,48.130993,4.761905,98.976109,"[[290, 3], [20, 1]]",0.754754,0.203742
3,rf_k_rf_k10,0.428571,0.866894,0.647733,35.226719,42.857143,86.689420,"[[254, 39], [12, 9]]",0.748253,0.218201
4,rf_k_rf_k30,0.238095,0.935154,0.586624,41.337559,23.809524,93.515358,"[[274, 19], [16, 5]]",0.743865,0.191094
5,mspc_rf,0.190476,0.921502,0.555989,44.401105,19.047619,92.150171,"[[270, 23], [17, 4]]",0.729725,0.141707
6,xgb_rf_k_k10,0.095238,0.982935,0.539087,46.091338,9.523810,98.293515,"[[288, 5], [19, 2]]",0.728913,0.190693
7,rf_k_lr_k15,0.666667,0.744027,0.705347,29.465301,66.666667,74.402730,"[[218, 75], [7, 14]]",0.726962,0.153422
8,xgb_rf_k_k15,0.047619,0.989761,0.518690,48.130993,4.761905,98.976109,"[[290, 3], [20, 1]]",0.722899,0.199776
9,xgb_rf_k_k30,0.095238,0.989761,0.542500,45.750041,9.523810,98.976109,"[[290, 3], [19, 2]]",0.718999,0.214724


In [ ]:
comparison = (
    leaderboard.merge(
        holdout,
        on="pipeline",
        suffixes=("_cv", "_holdout"),
    )[
        [
            "pipeline",
            "mean_roc_auc",
            "roc_auc",
            "mean_ber_percent",
            "ber_percent",
        ]
    ]
    .rename(
        columns={
            "mean_roc_auc": "roc_auc_cv",
            "roc_auc": "roc_auc_holdout",
            "mean_ber_percent": "ber_cv",
            "ber_percent": "ber_holdout",
        }
    )
    .sort_values("roc_auc_cv", ascending=False)
)
display(comparison)

,pipeline,roc_auc_cv,roc_auc_holdout,ber_cv,ber_holdout
0,rf_k_rf_k30,0.720646,0.743865,41.536639,41.337559
1,rf_k_rf_k20,0.714456,0.755891,40.202174,39.980497
2,rf_k_rf_k15,0.710502,0.758004,39.207328,40.833740
3,rf_k_rf_k10,0.708852,0.748253,39.423831,35.226719
4,mspc_rf,0.701939,0.729725,40.166730,44.401105
5,xgb_mspc,0.693545,0.700959,43.226496,49.666829
6,mspc_lr,0.689412,0.652852,36.824786,39.996750
7,xgb_rf_k_k30,0.685676,0.718999,47.285508,45.750041
8,xgb_rf_k_k20,0.682412,0.754754,46.634490,48.130993
9,rf_k_lr_k30,0.679981,0.692020,37.381222,33.373964


In [8]:
save_benchmark_results(
    tuned,
    leaderboard,
    holdout,
    train_rows=len(train_df),
    test_rows=len(test_df),
)
print(f"Wrote {BENCHMARK_RESULTS_PATH}")

Wrote /home/troy/SECOM/data/processed/secom_pipeline_benchmark.json


## Top features: `rf_k_rf`

Fit on train and rank **RandomForest** `feature_importances_` against preprocess output names (selected sensors, `hotelling_t2`, passthrough aux).

In [9]:
import pandas as pd

PIPELINE_NAME = "rf_k_rf"
rf_k30 = pipelines[PIPELINE_NAME]
rf_k30.fit(X_train, y_train)

feature_names = rf_k30.named_steps["preprocess"].get_feature_names_out()
importances = rf_k30.named_steps["classifier"].feature_importances_

top_features = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
top_features["importance_pct"] = 100 * top_features["importance"] / top_features["importance"].sum()

print(f"{PIPELINE_NAME}: top {len(top_features)} features by RF importance (train fit)")
display(top_features.head(50))

rf_k_rf_k30: top 66 features by RF importance (train fit)


,feature,importance,importance_pct
0,c_103,8.883588e-02,8.883588e+00
1,c_59,6.229465e-02,6.229465e+00
2,c_33,5.727914e-02,5.727914e+00
3,c_477,5.147411e-02,5.147411e+00
4,c_130,4.056453e-02,4.056453e+00
5,c_64,3.727688e-02,3.727688e+00
6,c_510,3.696125e-02,3.696125e+00
7,c_31,3.663876e-02,3.663876e+00
8,c_351,3.531168e-02,3.531168e+00
9,c_183,3.221827e-02,3.221827e+00


## Top features: `rf_k_lr`

Fit on train and rank **elastic-net logistic** coefficients (`|coef_|`) against preprocess output names (K=30 selected sensors, `hotelling_t2`, passthrough aux).

In [10]:
import numpy as np

PIPELINE_NAME = "rf_k_lr"
lr_k30 = pipelines[PIPELINE_NAME]
lr_k30.fit(X_train, y_train)

feature_names = lr_k30.named_steps["preprocess"].get_feature_names_out()
coefs = lr_k30.named_steps["classifier"].coef_.ravel()

top_features_lr = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "coefficient": coefs,
            "abs_coefficient": np.abs(coefs),
        }
    )
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)
top_features_lr["abs_coef_pct"] = (
    100 * top_features_lr["abs_coefficient"] / top_features_lr["abs_coefficient"].sum()
)

n_nonzero = int((top_features_lr["coefficient"] != 0).sum())
print(
    f"{PIPELINE_NAME}: {n_nonzero} / {len(top_features_lr)} non-zero coefs (train fit)"
)
display(top_features_lr.head(50))

rf_k_lr_k30: 44 / 66 non-zero coefs (train fit)


,feature,coefficient,abs_coefficient,abs_coef_pct
0,c_195__missing,1.435784,1.435784,9.891587
1,c_64,1.101271,1.101271,7.587017
2,c_121,1.061790,1.061790,7.315019
3,month_cos,0.853031,0.853031,5.876807
4,c_65,-0.718233,0.718233,4.948139
5,c_183,0.624397,0.624397,4.301676
6,c_58,0.603938,0.603938,4.160725
7,c_124,-0.587291,0.587291,4.046040
8,c_59,0.552028,0.552028,3.803101
9,c_129,0.464206,0.464206,3.198070
